In [ ]:
#| default_exp names

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import re
from functools import lru_cache
from fastcore.all import L
from litesearch.sanskrit import fold_token, DEVANAGARI
from ganapati.metre import deva2iast

## One key per name

A reader writes Krishna, Kṛṣṇa, krsna or कृष्ण; Bheeshma or Bhīṣma; Venkateshwara or Veṅkaṭeśvara. `name_key` gives them one key:
`fold_token`, with ṛ read as `ri`, `sh` as `s`, `w` as `v`, `ee` and `oo` as `i` and `u`, no aspirate `h`, and doubled letters single.
It is a key for matching, not a spelling to show.

In [ ]:
#| export
_ASP = re.compile(r'([bcdgjkpt])h')

@lru_cache(maxsize=1 << 15)
def name_key(s:str) -> str:
    'A romanisation-blind key for one Sanskrit name in IAST, plain roman, Indian-English or Devanagari: Bhīṣma, Bheeshma and भीष्म are `bisma`.'
    s = (s or '').strip().lower()
    if DEVANAGARI.search(s): s = deva2iast(s)
    s = fold_token(s.replace('ṛ', 'ri').replace('ṝ', 'ri'))
    s = s.replace('sh', 's').replace('w', 'v').replace('ee', 'i').replace('oo', 'u')
    return re.sub(r'(.)\1+', r'\1', re.sub('[^a-z]', '', _ASP.sub(r'\1', s)))

In [ ]:
assert name_key('Bhīṣma') == name_key('Bheeshma') == name_key('bhishma') == name_key('भीष्म') == 'bisma'
assert name_key('Kṛṣṇa') == name_key('Krishna') == name_key('कृष्ण') == 'krisna'
assert name_key('Veṅkaṭeśvara') == name_key('Venkateshwara') == name_key('Venkateswara')
assert name_key('Yudhiṣṭhira') == name_key('Yudhishthira') == 'yudistira'
assert name_key('Lakṣmaṇa') == name_key('Lakshmana') and name_key('Kārttikeya') == name_key('Kartikeya')
assert name_key('Bhīma') != name_key('Bhīṣma')
assert name_key('') == ''

## Spelled with a vowel more or less

Plain roman drops the vowel IAST writes as a diacritic (`krsna` for kṛṣṇa) and Indian-English adds one (`krishna`). `name_variants` finds
the keys in a pool, such as an index's vocabulary, that differ from a key by one vowel put in or left out. A consonant is never added:
Bhīma (`bima`) is not Bhīṣma (`bisma`).

In [ ]:
#| export
def _vowel1(a, b):
    if abs(len(a) - len(b)) != 1: return False
    if len(a) > len(b): a, b = b, a
    i = next((i for i, (x, y) in enumerate(zip(a, b)) if x != y), len(a))
    return b[i] in 'aeiou' and a[i:] == b[i + 1:]

def name_variants(key:str,       # a `name_key`, or any key the pool is keyed by
                  pool,          # the keys to look in: a set, a dict's keys, an index vocabulary
                  min_len:int=5  # shorter keys match only themselves; one vowel on a four-letter name is often another name
                  ) -> set:
    'The keys in `pool` that romanise the same name as `key`: itself, and from `min_len` letters those one vowel longer or shorter with the same first letter.'
    pool = set(pool)
    got = {key} & pool
    if len(key) >= min_len: got |= {p for p in pool if p[:1] == key[:1] and _vowel1(key, p)}
    return got

In [ ]:
pool = {'krsna', 'krisna', 'bima', 'bisma', 'arjuna', 'rama'}
assert name_variants('krsna', pool) == {'krsna', 'krisna'}
assert name_variants('krisna', pool) == {'krsna', 'krisna'}
assert name_variants('bisma', pool) == {'bisma'}           # not bima: a dropped consonant is another name
assert name_variants('arjun', pool) == {'arjuna'}          # Indian-English drops the final a
assert name_variants('ram', pool) == set()                 # under min_len: rama is not tried
assert name_variants('karisana', {'krsna'}) == set()       # three vowels

## Epithets

`EPITHETS` holds one group per line: the names a translation writes for a deity or figure, then `|`, then the epithets a reader may use.
The map is one way. Raṅganātha reaches Viṣṇu, Nārāyaṇa and Hari; Viṣṇu does not reach every epithet, so a search for Rāma is not widened to
every Rāghava. An epithet two figures share is in both groups (Acyuta, Mādhava). A name that is mostly another word is left out: Śrī,
Ramā (Lakṣmī, and `rama` is Rāma's key), Kṛṣṇā (Draupadī, and `krisna` is Kṛṣṇa's), Guha (Skanda, and the boatman of the Rāmāyaṇa), Kaunteya
(any son of Kuntī). Spellings that `name_key` cannot bring together are listed beside the IAST: Subramanya, Murugan, Bajrangbali.

The names are common knowledge from the epics, the Purāṇas and temple usage, written in IAST.

In [ ]:
#| export
EPITHETS = """
viṣṇu nārāyaṇa hari | raṅganātha raṅganāthar perumāḷ śrīnivāsa veṅkaṭeśa veṅkaṭeśvara veṅkaṭaramaṇa bālājī govindarāja padmanābha trivikrama jagannātha vaikuṇṭhanātha lakṣmīpati śrīdhara mukunda acyuta mādhava
kṛṣṇa | keśava govinda mādhava vāsudeva madhusūdana janārdana hṛṣīkeśa acyuta gopāla mukunda murāri pārthasārathi devakīnandana yogeśvara giridhārī muralīdhara kanhaiyā
śiva | śaṅkara mahādeva hara maheśvara rudra śambhu nīlakaṇṭha paśupati tryambaka candraśekhara gaṅgādhara naṭarāja viśveśvara umāpati sadāśiva īśāna śivan
pārvatī umā gaurī | ambikā bhavānī girijā aparṇā haimavatī śailaputrī kāmākṣī mīnākṣī
durgā | caṇḍikā kātyāyanī mahiṣāsuramardinī
lakṣmī | kamalā indirā mahālakṣmī śrīdevī padmāvatī
sarasvatī | śāradā vāgdevī vīṇāpāṇi
skanda kārttikeya subrahmaṇya | subramaṇya subramaniam murugan muruga ṣaṇmukha ṣaḍānana svāminātha senānī velan
gaṇeśa gaṇapati | vināyaka vinayagar piḷḷaiyār vighneśvara vighnarāja gajānana ekadanta lambodara heramba vakratuṇḍa gajavaktra
brahmā | caturmukha svayambhū kamalāsana virañci
indra | śakra maghavan purandara vāsava devarāja śacīpati vajrin
agni | vahni pāvaka hutāśana vaiśvānara jātavedas anala
sūrya | bhāskara ravi divākara dinakara mārtaṇḍa arka
vāyu | pavana anila
hanumān hanumat | māruti āñjaneya vāyuputra pavanaputra bajrangbalī kesarīnandana
rāma | rāghava dāśarathi rāmacandra kākutstha raghunandana raghupati raghunātha raghuvīra kosalendra
sītā | jānakī vaidehī maithilī bhūmijā
lakṣmaṇa | saumitri
rāvaṇa | daśagrīva daśānana daśakaṇṭha laṅkeśa laṅkeśvara
arjuna | pārtha dhanañjaya guḍākeśa savyasācin kirīṭin phalguna bībhatsu gāṇḍīvin śvetavāhana
yudhiṣṭhira | dharmarāja ajātaśatru dharmaputra
bhīma | vṛkodara bhīmasena
bhīṣma | gāṅgeya devavrata śāntanava
karṇa | rādheya vasuṣeṇa
draupadī | pāñcālī yājñasenī
droṇa | droṇācārya
duryodhana | suyodhana
vyāsa | vedavyāsa kṛṣṇadvaipāyana dvaipāyana bādarāyaṇa
"""

def _groups(t): return L(ln.split('|') for ln in t.strip().splitlines()).map(lambda g: (tuple(g[0].split()), tuple(g[1].split())))

def _one_way(groups):
    "Each name's key to the canonical names of its groups, epithets and canonical names apart."
    epi, can = {}, {}
    for cs, es in groups:
        for d, ws in ((can, cs), (epi, es)):
            for w in ws: d[name_key(w)] = tuple(dict.fromkeys(d.get(name_key(w), ()) + cs))
    return epi, can

_EPI, _CAN = _one_way(_groups(EPITHETS))

In [ ]:
#| export
def epithets() -> dict:
    'Every epithet in `EPITHETS` by `name_key`, to the canonical names a translation writes: `ranganata` gives Viṣṇu, Nārāyaṇa, Hari.'
    return dict(_EPI)

_HON, _KEYS = ('srimat', 'sri', 'bagavan'), set(_EPI) | set(_CAN)

def canonical(name:str) -> tuple:
    "The canonical names for `name`: an epithet's, a canonical name's own group, else `()`; allows a leading Śrī, a dropped final `a`, and one vowel more or less."
    k = name_key(name)
    ks = [k] + [k[len(h):] for h in _HON if k.startswith(h) and len(k) > len(h) + 2]
    for k in ks + [k + 'a' for k in ks]:
        if (r := _EPI.get(k) or _CAN.get(k)): return r
    vs = sorted(v for k in ks for v in name_variants(k, _KEYS))
    return tuple(dict.fromkeys(c for v in vs for c in _EPI.get(v) or _CAN[v]))

In [ ]:
assert canonical('Raṅganātha') == ('viṣṇu', 'nārāyaṇa', 'hari') == canonical('Sri Ranganatha') == canonical('Srinivasa')
assert canonical('Venkateshwara') == canonical('Balaji') == canonical('perumal') == ('viṣṇu', 'nārāyaṇa', 'hari')
assert canonical('Nārāyaṇa') == ('viṣṇu', 'nārāyaṇa', 'hari')              # a canonical name gives its group
assert canonical('Dāśarathi') == canonical('Raghava') == canonical('dasarathi') == ('rāma',)
assert canonical('Partha') == canonical('Dhananjaya') == canonical('धनञ्जय') == ('arjuna',)
assert canonical('Jānakī') == canonical('Vaidehi') == ('sītā',) and canonical('Maruti') == canonical('Anjaneya') == ('hanumān', 'hanumat')
assert canonical('Murugan') == canonical('Subramanya') == ('skanda', 'kārttikeya', 'subrahmaṇya')
assert canonical('Vinayaka') == canonical('Ganesh') == ('gaṇeśa', 'gaṇapati')    # Ganesh: the final a dropped
assert canonical('Shankar') == canonical('Mahadeva') == ('śiva',) and canonical('Ambika') == ('pārvatī', 'umā', 'gaurī')
assert set(canonical('Acyuta')) == {'viṣṇu', 'nārāyaṇa', 'hari', 'kṛṣṇa'}       # an epithet two groups share
assert canonical('Keśava') == canonical('Kesava') == ('kṛṣṇa',) and canonical('Krishna') == canonical('krsna') == ('kṛṣṇa',)   # krsna: one vowel
assert canonical('Gangeya') == ('bhīṣma',) and canonical('Vrkodara') == ('bhīma',)
assert canonical('dharma') == canonical('karma') == canonical('') == ()

No epithet's key may be another group's canonical name, or a search for that name would be widened: the table is checked for it.

In [ ]:
e = epithets()
assert e['ranganata'] == ('viṣṇu', 'nārāyaṇa', 'hari') and 'visnu' not in e   # one way: canonical names are not keys
assert not [k for k, cs in e.items() if k in _CAN and not set(cs) & set(_CAN[k])]
assert len(e) > 150 and len(_CAN) > 30

## Technical terms

`TERMS` gives the English a translation uses for a śāstra term: the pramāṇas and the Nyāya categories, the Vaiśeṣika padārthas, Sāṃkhya and
Yoga, Vedānta and the Gītā, and the pañcāṅga and horoscope. Each gloss is a few words, most usual first. `term_gloss` reads it by `name_key`,
so anumāna, anumana and अनुमान are one term. A term not in the table falls back to Monier-Williams (`mw_lexicon`) when that is passed or
already downloaded; nothing is downloaded here.

In [ ]:
#| export
TERMS = """
pramāṇa: means of knowledge, proof
pratyakṣa: perception
anumāna: inference
upamāna: comparison, analogy
śabda: testimony
arthāpatti: presumption, postulation
anupalabdhi: non-apprehension
prameya: object of knowledge
saṃśaya: doubt
prayojana: purpose
dṛṣṭānta: example
siddhānta: established tenet, conclusion
avayava: member of a syllogism
tarka: hypothetical reasoning
nirṇaya: ascertainment
vāda: discussion
jalpa: wrangling
vitaṇḍā: cavil
hetvābhāsa: fallacy
chala: quibble
nigrahasthāna: point of defeat
pratijñā: proposition
hetu: reason
udāharaṇa: example
upanaya: application
nigamana: conclusion
vyāpti: invariable concomitance
sādhya: what is to be proved
padārtha: category
dravya: substance
guṇa: quality
karma: action
sāmānya: generality
viśeṣa: particularity
samavāya: inherence
abhāva: non-existence, absence
paramāṇu: atom
puruṣa: spirit, person
prakṛti: primal nature, matter
sattva: goodness, purity
rajas: passion, activity
tamas: darkness, inertia
mahat: the great principle
ahaṃkāra: ego, I-maker
buddhi: intellect
manas: mind
tanmātra: subtle element
tattva: principle
kaivalya: isolation
citta: mind
vṛtti: fluctuation, modification
abhyāsa: practice
vairāgya: dispassion, detachment
kleśa: affliction
avidyā: ignorance
asmitā: egoism
rāga: attachment
dveṣa: aversion
abhiniveśa: clinging to life
niyama: observance
āsana: posture
prāṇāyāma: breath control
pratyāhāra: withdrawal of the senses
dhāraṇā: concentration
dhyāna: meditation
samādhi: absorption
ahiṃsā: nonviolence
satya: truth
asteya: non-stealing
brahmacarya: celibacy
aparigraha: non-possessiveness
tapas: austerity
svādhyāya: self-study
santoṣa: contentment
śauca: purity, cleanliness
brahman: the absolute
ātman: self
jīva: individual soul
māyā: illusion
mokṣa: liberation, release
mukti: liberation
saṃsāra: cycle of rebirth
dharma: duty, righteousness
svadharma: one's own duty
jñāna: knowledge
bhakti: devotion
śraddhā: faith
yajña: sacrifice
sthitaprajña: steady wisdom
niṣkāma: desireless
prapatti: surrender
śaraṇāgati: surrender, refuge
ānanda: bliss
viveka: discrimination
adhyāsa: superimposition
upādhi: limiting adjunct
avatāra: incarnation, descent
ṛṣi: seer
chandas: metre
devatā: deity
graha: planet
rāśi: sign
nakṣatra: lunar mansion, asterism
bhāva: house
lagna: ascendant
tithi: lunar day
karaṇa: half lunar day
daśā: planetary period
antardaśā: sub-period
gocara: transit
ayanāṃśa: precession correction
drekkāṇa: decanate
navāṃśa: ninth division
dṛṣṭi: aspect
uccha: exaltation
nīca: debilitation
vakra: retrograde
kendra: angle, quadrant
trikoṇa: trine
pañcāṅga: almanac
muhūrta: auspicious time
grahaṇa: eclipse
saṅkrānti: solar ingress
adhimāsa: intercalary month
uttarāyaṇa: northward course of the sun
dakṣiṇāyana: southward course of the sun
"""

_TERMS = {name_key(t): tuple(g.strip() for g in gs.split(',')) for t, gs in (ln.split(':', 1) for ln in TERMS.strip().splitlines())}

def terms() -> dict:
    'Every term in `TERMS` by `name_key`, to its English glosses.'
    return dict(_TERMS)

@lru_cache(maxsize=1)
def _mw_local():
    from ganapati.lemma import sanskrit_home, mw_lexicon
    return mw_lexicon() if sanskrit_home('mw.tsv').exists() else {}

def term_gloss(term:str,   # IAST, plain roman or Devanagari
               mw:dict=None # a `mw_lexicon()`; None uses the downloaded one if there is one
               ) -> tuple:
    'Short English for a Sanskrit technical term: `TERMS` first, then Monier-Williams for IAST or Devanagari; `()` when neither knows it.'
    if (g := _TERMS.get(name_key(term))): return g
    if not (mw := _mw_local() if mw is None else mw) or not term: return ()
    from ganapati.lemma import to_slp1
    return tuple(s for x in (mw.get(to_slp1(term.strip())) or '').split(';') if (s := x.strip()))[:3]

In [ ]:
assert term_gloss('anumāna') == term_gloss('anumana') == term_gloss('अनुमान') == ('inference',)
assert term_gloss('pratyaksha') == ('perception',) and term_gloss('upamana') == ('comparison', 'analogy')
assert term_gloss('saṃśaya') == term_gloss('samshaya') == ('doubt',) and term_gloss('dravya') == ('substance',)
assert term_gloss('sthitaprajña') == ('steady wisdom',) and term_gloss('moksha') == ('liberation', 'release')
assert term_gloss('nakshatra') == ('lunar mansion', 'asterism') and term_gloss('lagna') == ('ascendant',)
assert term_gloss('anumāna', mw={}) == ('inference',)                                   # the table needs no MW
assert term_gloss('gaja', mw={'gaja': 'an elephant; the number 8'}) == ('an elephant', 'the number 8')
assert term_gloss('xyz', mw={}) == () and term_gloss('', mw={}) == ()
assert len(_TERMS) == len(TERMS.strip().splitlines()) > 120                           # no two terms share a key

Terms and epithets are kept apart: no term is a name.

In [ ]:
assert not set(_TERMS) & (set(_EPI) | set(_CAN))

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()